In [ ]:
import pandas as pd

df = pd.read_csv("https://s3.amazonaws.com/google-landmark/metadata/train.csv")
print(df.shape)
df.head()

(4132914, 3)


,id,url,landmark_id
0,6e158a47eb2ca3f6,https://upload.wikimedia.org/wikipedia/commons...,142820
1,202cd79556f30760,http://upload.wikimedia.org/wikipedia/commons/...,104169
2,3ad87684c99c06e1,http://upload.wikimedia.org/wikipedia/commons/...,37914
3,e7f70e9c61e66af3,https://upload.wikimedia.org/wikipedia/commons...,102140
4,4072182eddd0100e,https://upload.wikimedia.org/wikipedia/commons...,2474


In [ ]:
samples = 20000
df = df.loc[:samples, :]
num_classes = len(df["landmark_id"].unique())
print(num_classes)

16342


In [ ]:
counts = df["landmark_id"].value_counts()
valid_classes = counts[counts >= 5].index
df = df[df["landmark_id"].isin(valid_classes)].reset_index(drop=True)
num_classes = len(df["landmark_id"].unique())
print(f"Remaining samples: {len(df)}, classes: {num_classes}")

Remaining samples: 733, classes: 100


In [ ]:
from sklearn.preprocessing import LabelEncoder

lencoder = LabelEncoder()
df["label"] = lencoder.fit_transform(df["landmark_id"])

In [ ]:
import requests, os
from concurrent.futures import ThreadPoolExecutor

os.makedirs("images", exist_ok=True)

def download_image(row):
    path = f"images/{row['id']}.jpg"
    if os.path.exists(path):
        return
    try:
        img_data = requests.get(row["url"], timeout=5).content
        with open(path, "wb") as f:
            f.write(img_data)
    except Exception:
        pass

with ThreadPoolExecutor(max_workers=20) as executor:
    list(executor.map(download_image, [row for _, row in df.iterrows()]))

In [ ]:
import os
print(len(os.listdir("images")))

733


In [ ]:
df["path"] = df["id"].apply(lambda x: f"images/{x}.jpg")
df = df[df["path"].apply(os.path.exists)].reset_index(drop=True)
print(f"Final usable samples: {len(df)}")

Final usable samples: 733


In [ ]:
import cv2
import numpy as np

def get_batch(df, start, batch_size):
    batch_df = df.iloc[start:start+batch_size]
    images = []
    labels = []
    for _, row in batch_df.iterrows():
        img = cv2.imread(row["path"])
        if img is not None:
            img = cv2.resize(img, (224, 224))
            images.append(img)
            labels.append(row["label"])
    return np.array(images), np.array(labels)

In [ ]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
print(len(train_df), len(val_df))

586 147


In [ ]:
from tensorflow.keras.applications.vgg19 import VGG19
from tensorflow.keras.layers import Dense
from tensorflow.keras.models import Sequential

source_model = VGG19(weights="imagenet", include_top=False, input_shape=(224,224,3), pooling="avg")

model = Sequential()
model.add(source_model)
model.add(Dense(num_classes, activation="softmax"))

model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

80134624/80134624 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ vgg19 (Functional)              │ (None, 512)            │    20,024,384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 100)            │        51,300 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,075,684 (76.58 MB)

 Trainable params: 20,075,684 (76.58 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
batch_size = 16
epochs = 5

for epoch in range(epochs):
    for it in range(int(np.ceil(len(train_df)/batch_size))):
        X_batch, y_batch = get_batch(train_df, it*batch_size, batch_size)
        if len(X_batch) > 0:
            model.train_on_batch(X_batch, y_batch)
    print(f"Epoch {epoch+1} done")

Epoch 1 done
Epoch 2 done
Epoch 3 done
Epoch 4 done
Epoch 5 done


In [ ]:
errors = 0
good_preds = []
bad_preds = []

for it in range(int(np.ceil(len(val_df)/batch_size))):
    X_val, y_val = get_batch(val_df, it*batch_size, batch_size)
    if len(X_val) == 0:
        continue
    result = model.predict(X_val)
    cla = np.argmax(result, axis=1)
    for idx, res in enumerate(result):
        if cla[idx] != y_val[idx]:
            errors += 1
            bad_preds.append([batch_size*it + idx, cla[idx], res[cla[idx]]])
        else:
            good_preds.append([batch_size*it + idx, cla[idx], res[cla[idx]]])

print(f"Errors: {errors} / {len(val_df)}")

Errors: 0 / 147


In [ ]:
import numpy as np

all_preds = [p[1] for p in good_preds] + [p[1] for p in bad_preds]
print("Unique predicted classes used:", len(set(all_preds)))
print("Sample predictions:", all_preds[:20])

Unique predicted classes used: 0
Sample predictions: []


In [ ]:
train_errors = 0
for it in range(int(np.ceil(len(train_df)/batch_size))):
    X_t, y_t = get_batch(train_df, it*batch_size, batch_size)
    if len(X_t) == 0:
        continue
    result = model.predict(X_t, verbose=0)
    cla = np.argmax(result, axis=1)
    for idx in range(len(cla)):
        if cla[idx] != y_t[idx]:
            train_errors += 1
print(f"Train errors: {train_errors} / {len(train_df)}")

Train errors: 0 / 586


In [ ]:
import cv2

sample_path = df.iloc[0]["path"]
print(sample_path)
img = cv2.imread(sample_path)
print(img)

images/ccf070a2da242d56.jpg
None


In [ ]:
import os

sizes = [os.path.getsize(p) for p in df["path"]]
print(min(sizes), max(sizes), sum(sizes)/len(sizes))

126 126 126.0


In [ ]:
def is_valid_image(path):
    img = cv2.imread(path)
    return img is not None

df["valid"] = df["path"].apply(is_valid_image)
print(df["valid"].value_counts())

df = df[df["valid"]].reset_index(drop=True)
print(f"Truly usable samples: {len(df)}")

valid
False    733
Name: count, dtype: int64
Truly usable samples: 0


In [ ]:
import requests, os
from concurrent.futures import ThreadPoolExecutor

os.makedirs("images", exist_ok=True)

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36"
}

def download_image(row):
    path = f"images/{row['id']}.jpg"
    try:
        resp = requests.get(row["url"], headers=headers, timeout=5)
        if resp.status_code == 200 and len(resp.content) > 1000:  # basic sanity check
            with open(path, "wb") as f:
                f.write(resp.content)
    except Exception:
        pass

with ThreadPoolExecutor(max_workers=20) as executor:
    list(executor.map(download_image, [row for _, row in df.iterrows()]))

In [ ]:
import shutil
shutil.rmtree("images")
os.makedirs("images", exist_ok=True)

In [ ]:
sizes = [os.path.getsize(f"images/{f}") for f in os.listdir("images")]
print(len(sizes), min(sizes) if sizes else None, max(sizes) if sizes else None)

0 None None


In [ ]:
df["path"] = df["id"].apply(lambda x: f"images/{x}.jpg")
df["valid"] = df["path"].apply(lambda p: os.path.exists(p) and cv2.imread(p) is not None)
print(df["valid"].value_counts())
df = df[df["valid"]].reset_index(drop=True)
print(f"Truly usable samples: {len(df)}")

Series([], Name: count, dtype: int64)
Truly usable samples: 0


In [ ]:
import pandas as pd
import requests, os, shutil
from concurrent.futures import ThreadPoolExecutor
from sklearn.preprocessing import LabelEncoder
import cv2

# 1. Reload real data
df = pd.read_csv("https://s3.amazonaws.com/google-landmark/metadata/train.csv")
samples = 20000
df = df.loc[:samples, :]

counts = df["landmark_id"].value_counts()
valid_classes = counts[counts >= 5].index
df = df[df["landmark_id"].isin(valid_classes)].reset_index(drop=True)

lencoder = LabelEncoder()
df["label"] = lencoder.fit_transform(df["landmark_id"])
print("Step 1 - filtered rows:", len(df))

# 2. Fresh download folder
if os.path.exists("images"):
    shutil.rmtree("images")
os.makedirs("images", exist_ok=True)

headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36"}

# 3. Test ONE download first, print everything
test_row = df.iloc[0]
print("Testing URL:", test_row["url"])
try:
    resp = requests.get(test_row["url"], headers=headers, timeout=10)
    print("Status code:", resp.status_code)
    print("Content length:", len(resp.content))
    print("First 100 bytes:", resp.content[:100])
except Exception as e:
    print("ERROR:", e)

Step 1 - filtered rows: 733
Testing URL: https://upload.wikimedia.org/wikipedia/commons/e/e5/ETH-BIB-Z%C3%BCrich%2C_ETH_Z%C3%BCrich%2C_Chemiegeb%C3%A4ude_%28CAB%29%2C_Westfassade-Ans_00381-F.tif
Status code: 429
Content length: 1962
First 100 bytes: b'<!DOCTYPE html>\n<html lang="en">\n<meta charset="utf-8">\n<title>Wikimedia Error</title>\n<style>\n* { m'


In [ ]:
import time

def download_image(row, headers):
    path = f"images/{row['id']}.jpg"
    try:
        resp = requests.get(row["url"], headers=headers, timeout=10)
        if resp.status_code == 200 and len(resp.content) > 1000:
            # Skip non-JPEG formats (like .tif) since cv2 often can't read them reliably
            if row["url"].lower().endswith((".tif", ".tiff", ".svg", ".gif")):
                return False
            with open(path, "wb") as f:
                f.write(resp.content)
            return True
        return False
    except Exception:
        return False

# Rebuild folder
if os.path.exists("images"):
    shutil.rmtree("images")
os.makedirs("images", exist_ok=True)

headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36"}

success_count = 0
for i, row in df.iterrows():
    if download_image(row, headers):
        success_count += 1
    time.sleep(0.3)  # slow down to avoid rate-limiting
    if i % 50 == 0:
        print(f"Processed {i}, successes so far: {success_count}")

print("Total successful downloads:", success_count)

Processed 0, successes so far: 0
Processed 50, successes so far: 0
Processed 100, successes so far: 0
Processed 150, successes so far: 0
Processed 200, successes so far: 0
Processed 250, successes so far: 0
Processed 300, successes so far: 0
Processed 350, successes so far: 0
Processed 400, successes so far: 0
Processed 450, successes so far: 0
Processed 500, successes so far: 0
Processed 550, successes so far: 0
Processed 600, successes so far: 0
Processed 650, successes so far: 0
Processed 700, successes so far: 0
Total successful downloads: 0


In [ ]:
import urllib.request

url = "https://s3.amazonaws.com/google-landmark/train/images_000.tar"
urllib.request.urlretrieve(url, "images_000.tar")
print("Download complete")

Download complete


In [ ]:
import tarfile

with tarfile.open("images_000.tar") as tar:
    tar.extractall("train_images")

print("Extraction complete")

/tmp/ipykernel_1140/2420060136.py:4: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall("train_images")


Extraction complete


In [ ]:
import os

count = 0
for root, dirs, files in os.walk("train_images"):
    count += len(files)
print("Total images extracted:", count)

# peek at folder structure
for root, dirs, files in os.walk("train_images"):
    if files:
        print(root, files[:3])
        break

Total images extracted: 8266
train_images/0/0/3 ['00380a7c9739dbd5.jpg', '0038d0560770acb2.jpg', '003061b6ddab9cdd.jpg']


In [ ]:
import pandas as pd
import glob

# Get list of extracted image ids (filename without .jpg)
extracted_paths = glob.glob("train_images/**/*.jpg", recursive=True)
extracted_ids = [os.path.basename(p).replace(".jpg", "") for p in extracted_paths]
print("Extracted images:", len(extracted_ids))

# Load full metadata and filter to only images you actually have
df_full = pd.read_csv("https://s3.amazonaws.com/google-landmark/metadata/train.csv")
df = df_full[df_full["id"].isin(extracted_ids)].reset_index(drop=True)

# Map id -> actual file path
id_to_path = {os.path.basename(p).replace(".jpg",""): p for p in extracted_paths}
df["path"] = df["id"].map(id_to_path)

print("Matched samples:", len(df))

Extracted images: 8266
Matched samples: 8266


In [ ]:
counts = df["landmark_id"].value_counts()
valid_classes = counts[counts >= 5].index
df = df[df["landmark_id"].isin(valid_classes)].reset_index(drop=True)

from sklearn.preprocessing import LabelEncoder
lencoder = LabelEncoder()
df["label"] = lencoder.fit_transform(df["landmark_id"])
num_classes = len(df["label"].unique())
print(f"Final usable samples: {len(df)}, classes: {num_classes}")

Final usable samples: 95, classes: 12


In [ ]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
print("Train:", len(train_df), "Val:", len(val_df))

Train: 76 Val: 19


In [ ]:
import cv2
import numpy as np

def get_batch(df, start, batch_size):
    batch_df = df.iloc[start:start+batch_size]
    images = []
    labels = []
    for _, row in batch_df.iterrows():
        img = cv2.imread(row["path"])
        if img is not None:
            img = cv2.resize(img, (224, 224))
            images.append(img)
            labels.append(row["label"])
    return np.array(images), np.array(labels)

In [ ]:
from tensorflow.keras.applications.vgg19 import VGG19
from tensorflow.keras.layers import Dense
from tensorflow.keras.models import Sequential

source_model = VGG19(weights="imagenet", include_top=False, input_shape=(224,224,3), pooling="avg")

model = Sequential()
model.add(source_model)
model.add(Dense(num_classes, activation="softmax"))

model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

80134624/80134624 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ vgg19 (Functional)              │ (None, 512)            │    20,024,384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 12)             │         6,156 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,030,540 (76.41 MB)

 Trainable params: 20,030,540 (76.41 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
batch_size = 16
epochs = 5

for epoch in range(epochs):
    for it in range(int(np.ceil(len(train_df)/batch_size))):
        X_batch, y_batch = get_batch(train_df, it*batch_size, batch_size)
        if len(X_batch) > 0:
            model.train_on_batch(X_batch, y_batch)
    print(f"Epoch {epoch+1} done")

Epoch 1 done
Epoch 2 done
Epoch 3 done
Epoch 4 done
Epoch 5 done


In [ ]:
errors = 0
good_preds = []
bad_preds = []

for it in range(int(np.ceil(len(val_df)/batch_size))):
    X_val, y_val = get_batch(val_df, it*batch_size, batch_size)
    if len(X_val) == 0:
        continue
    result = model.predict(X_val, verbose=0)
    cla = np.argmax(result, axis=1)
    for idx in range(len(cla)):
        if cla[idx] != y_val[idx]:
            errors += 1
            bad_preds.append([batch_size*it + idx, cla[idx], result[idx][cla[idx]]])
        else:
            good_preds.append([batch_size*it + idx, cla[idx], result[idx][cla[idx]]])

print(f"Errors: {errors} / {len(val_df)}")

Errors: 15 / 19


In [ ]:
print("Good preds:", len(good_preds))
print("Bad preds:", len(bad_preds))
print("Sample good:", good_preds[:3])
print("Sample bad:", bad_preds[:3])

Good preds: 4
Bad preds: 15
Sample good: [[4, np.int64(7), np.float32(0.45126384)], [9, np.int64(7), np.float32(0.4333771)], [11, np.int64(7), np.float32(0.4032986)]]
Sample bad: [[0, np.int64(7), np.float32(0.25832325)], [1, np.int64(7), np.float32(0.24776174)], [2, np.int64(7), np.float32(0.2367505)]]


In [ ]:
import urllib.request

tar_files = ["images_001.tar", "images_002.tar", "images_003.tar", "images_004.tar"]

for fname in tar_files:
    url = f"https://s3.amazonaws.com/google-landmark/train/{fname}"
    print(f"Downloading {fname}...")
    urllib.request.urlretrieve(url, fname)
    print(f"{fname} done")

images_001.tar done
images_002.tar done
images_003.tar done
images_004.tar done


In [ ]:
import tarfile

for fname in tar_files:
    with tarfile.open(fname) as tar:
        tar.extractall("train_images")
    print(f"Extracted {fname}")

/tmp/ipykernel_833/1220977878.py:5: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall("train_images")


Extracted images_001.tar
Extracted images_002.tar
Extracted images_003.tar
Extracted images_004.tar


In [ ]:
import os

count = 0
for root, dirs, files in os.walk("train_images"):
    count += len(files)
print("Total images extracted:", count)

Total images extracted: 33064


In [ ]:
import pandas as pd
import glob

extracted_paths = glob.glob("train_images/**/*.jpg", recursive=True)
extracted_ids = [os.path.basename(p).replace(".jpg", "") for p in extracted_paths]
print("Extracted images:", len(extracted_ids))

df_full = pd.read_csv("https://s3.amazonaws.com/google-landmark/metadata/train.csv")
df = df_full[df_full["id"].isin(extracted_ids)].reset_index(drop=True)

id_to_path = {os.path.basename(p).replace(".jpg",""): p for p in extracted_paths}
df["path"] = df["id"].map(id_to_path)

print("Matched samples:", len(df))

Extracted images: 33064
Matched samples: 33064


In [ ]:
from sklearn.preprocessing import LabelEncoder

counts = df["landmark_id"].value_counts()
valid_classes = counts[counts >= 15].index   # stricter threshold now
df = df[df["landmark_id"].isin(valid_classes)].reset_index(drop=True)

# cap each class so no single class dominates
df = df.groupby("landmark_id").head(30).reset_index(drop=True)

lencoder = LabelEncoder()
df["label"] = lencoder.fit_transform(df["landmark_id"])
num_classes = len(df["label"].unique())
print(f"Final usable samples: {len(df)}, classes: {num_classes}")
print(df["landmark_id"].value_counts())

Final usable samples: 296, classes: 15
landmark_id
138982    30
62798     30
177870    27
176528    25
83144     20
192931    19
20409     18
161902    17
33992     17
126637    16
41808     16
84689     16
139894    15
194914    15
45428     15
Name: count, dtype: int64


In [ ]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["label"])
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
print("Train:", len(train_df), "Val:", len(val_df))

Train: 236 Val: 60


In [ ]:
import cv2
import numpy as np

def get_batch(df, start, batch_size):
    batch_df = df.iloc[start:start+batch_size]
    images = []
    labels = []
    for _, row in batch_df.iterrows():
        img = cv2.imread(row["path"])
        if img is not None:
            img = cv2.resize(img, (224, 224))
            images.append(img)
            labels.append(row["label"])
    return np.array(images), np.array(labels)

In [ ]:
from tensorflow.keras.applications.vgg19 import VGG19
from tensorflow.keras.layers import Dense
from tensorflow.keras.models import Sequential

source_model = VGG19(weights="imagenet", include_top=False, input_shape=(224,224,3), pooling="avg")

model = Sequential()
model.add(source_model)
model.add(Dense(num_classes, activation="softmax"))

model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

80134624/80134624 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ vgg19 (Functional)              │ (None, 512)            │    20,024,384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 15)             │         7,695 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,032,079 (76.42 MB)

 Trainable params: 20,032,079 (76.42 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
batch_size = 16
epochs = 8

for epoch in range(epochs):
    train_df = train_df.sample(frac=1).reset_index(drop=True)  # shuffle every epoch
    for it in range(int(np.ceil(len(train_df)/batch_size))):
        X_batch, y_batch = get_batch(train_df, it*batch_size, batch_size)
        if len(X_batch) > 0:
            model.train_on_batch(X_batch, y_batch)
    print(f"Epoch {epoch+1} done")

Epoch 1 done
Epoch 2 done
Epoch 3 done
Epoch 4 done
Epoch 5 done
Epoch 6 done
Epoch 7 done
Epoch 8 done


In [ ]:
errors = 0
good_preds = []
bad_preds = []

for it in range(int(np.ceil(len(val_df)/batch_size))):
    X_val, y_val = get_batch(val_df, it*batch_size, batch_size)
    if len(X_val) == 0:
        continue
    result = model.predict(X_val, verbose=0)
    cla = np.argmax(result, axis=1)
    for idx in range(len(cla)):
        if cla[idx] != y_val[idx]:
            errors += 1
            bad_preds.append([batch_size*it + idx, cla[idx], result[idx][cla[idx]]])
        else:
            good_preds.append([batch_size*it + idx, cla[idx], result[idx][cla[idx]]])

print(f"Errors: {errors} / {len(val_df)}")

all_preds = [p[1] for p in good_preds] + [p[1] for p in bad_preds]
print("Unique predicted classes used:", len(set(all_preds)))
print("Prediction distribution:", pd.Series(all_preds).value_counts())

Errors: 54 / 60
Unique predicted classes used: 2
Prediction distribution: 8     59
12     1
Name: count, dtype: int64


In [ ]:
from tensorflow.keras.applications.vgg19 import VGG19, preprocess_input
from tensorflow.keras.layers import Dense
from tensorflow.keras.models import Sequential

source_model = VGG19(weights="imagenet", include_top=False, input_shape=(224,224,3), pooling="avg")
source_model.trainable = False  # FREEZE the pretrained base

model = Sequential()
model.add(source_model)
model.add(Dense(num_classes, activation="softmax"))

model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ vgg19 (Functional)              │ (None, 512)            │    20,024,384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 15)             │         7,695 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,032,079 (76.42 MB)

 Trainable params: 7,695 (30.06 KB)

 Non-trainable params: 20,024,384 (76.39 MB)

In [ ]:
import cv2
import numpy as np

def get_batch(df, start, batch_size):
    batch_df = df.iloc[start:start+batch_size]
    images = []
    labels = []
    for _, row in batch_df.iterrows():
        img = cv2.imread(row["path"])
        if img is not None:
            img = cv2.resize(img, (224, 224))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  # VGG expects RGB
            images.append(img)
            labels.append(row["label"])
    images = np.array(images, dtype=np.float32)
    if len(images) > 0:
        images = preprocess_input(images)  # proper VGG19 normalization
    return images, np.array(labels)

In [ ]:
batch_size = 16
epochs = 10

for epoch in range(epochs):
    train_df = train_df.sample(frac=1).reset_index(drop=True)
    for it in range(int(np.ceil(len(train_df)/batch_size))):
        X_batch, y_batch = get_batch(train_df, it*batch_size, batch_size)
        if len(X_batch) > 0:
            model.train_on_batch(X_batch, y_batch)
    print(f"Epoch {epoch+1} done")

Epoch 1 done
Epoch 2 done
Epoch 3 done
Epoch 4 done
Epoch 5 done
Epoch 6 done
Epoch 7 done
Epoch 8 done
Epoch 9 done
Epoch 10 done


In [ ]:
errors = 0
good_preds = []
bad_preds = []

for it in range(int(np.ceil(len(val_df)/batch_size))):
    X_val, y_val = get_batch(val_df, it*batch_size, batch_size)
    if len(X_val) == 0:
        continue
    result = model.predict(X_val, verbose=0)
    cla = np.argmax(result, axis=1)
    for idx in range(len(cla)):
        if cla[idx] != y_val[idx]:
            errors += 1
            bad_preds.append([batch_size*it + idx, cla[idx], result[idx][cla[idx]]])
        else:
            good_preds.append([batch_size*it + idx, cla[idx], result[idx][cla[idx]]])

print(f"Errors: {errors} / {len(val_df)}")

import pandas as pd
all_preds = [p[1] for p in good_preds] + [p[1] for p in bad_preds]
print("Unique predicted classes used:", len(set(all_preds)))
print("Prediction distribution:", pd.Series(all_preds).value_counts())

Errors: 36 / 60
Unique predicted classes used: 15
Prediction distribution: 12    8
5     8
4     5
13    5
1     5
3     4
7     4
0     3
14    3
9     3
8     3
11    3
2     3
10    2
6     1
Name: count, dtype: int64
